# Prophet Training For Colab

This notebook mirrors `src/prophet/train_prophet.py`, but it also persists **per-ticker fitted Prophet models** so the output is reusable from `pipeline.predict` later.

Estimated training time on the current dataset:
- Prophet does **not** use the Colab GPU; it is CPU-bound
- Full dataset, `3` CV folds + final per-ticker fit: about `1-3.5 hours`
- `max_tickers=100`: often `8-25 minutes`
- `max_tickers=250`: often `20-60 minutes`

If you mainly want a pipeline-ready baseline quickly, start with `max_tickers=100` and scale up.


In [ ]:
# Runtime -> Change runtime type -> Hardware accelerator -> GPU
!pip -q install prophet xgboost torch scikit-learn pandas numpy joblib


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/diploma')  # Change if your repo is elsewhere in Colab or Drive.
DATASET_PATH = PROJECT_DIR / 'dataset' / 'stock_prices_20y.db'
MODELS_DIR = PROJECT_DIR / 'pipeline' / 'models'
RESULTS_DIR = PROJECT_DIR / 'results'
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'

for path in [MODELS_DIR, RESULTS_DIR, ARTIFACTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Missing dataset at {DATASET_PATH}. Update PROJECT_DIR or place the repo there first.'
    )

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('Project dir :', PROJECT_DIR)
print('Dataset path :', DATASET_PATH)


In [ ]:
import sqlite3
import time
from datetime import datetime

import numpy as np
import pandas as pd

with sqlite3.connect(DATASET_PATH) as conn:
    row_count = conn.execute('SELECT COUNT(*) FROM prices').fetchone()[0]
    ticker_count = conn.execute('SELECT COUNT(DISTINCT ticker) FROM prices').fetchone()[0]
    min_date, max_date = conn.execute('SELECT MIN(date), MAX(date) FROM prices').fetchone()

print(f'Rows: {row_count:,} | Tickers: {ticker_count:,} | Date range: {min_date} -> {max_date}')


In [ ]:
import json
import pickle
from dataclasses import asdict

from joblib import Parallel, delayed
from prophet.serialize import model_to_json

from src.prophet.train_prophet import (
    Config,
    add_features,
    build_expanding_time_folds,
    fit_prophet_for_ticker,
    load_prices_from_sqlite,
    predict_prophet_for_ticker,
    regression_metrics_with_direction,
    split_train_test_per_ticker,
)

RUN_TAG = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
REGRESSOR_COLS = ['log_volume', 'log_ret_1', 'hl_range', 'rsi_14']
N_WORKERS = min(4, os.cpu_count() or 1)

cfg = Config(
    db_path=str(DATASET_PATH),
    model_output_path=str(MODELS_DIR / f'{RUN_TAG}-prophet.pkl'),
    config_output_path=str(ARTIFACTS_DIR / f'{RUN_TAG}-prophet_config.json'),
)
print(cfg)
print('Prophet workers:', N_WORKERS)


In [ ]:
def run_prophet_for_ticker(train_ticker_df, eval_ticker_df, cfg, store_model=False):
    ticker = str(train_ticker_df['ticker'].iloc[0])
    try:
        model = fit_prophet_for_ticker(train_ticker_df, cfg)
        pred_close = predict_prophet_for_ticker(model, eval_ticker_df, cfg)
        actual_close = eval_ticker_df['close'].to_numpy(dtype=np.float64)
        last_train_close = float(train_ticker_df['close'].iloc[-1])
        prev_close = np.concatenate([[last_train_close], actual_close[:-1]])
        result = {
            'ticker': ticker,
            'pred_close': pred_close,
            'actual_close': actual_close,
            'prev_close': prev_close,
        }
        if store_model:
            result['model_json'] = model_to_json(model)
        return result
    except Exception as exc:
        return {'ticker': ticker, 'error': str(exc)}


In [ ]:
start_time = time.perf_counter()

print('Loading data...')
raw_df = load_prices_from_sqlite(cfg.db_path, cfg.table_name)
raw_df = add_features(raw_df)
all_tickers = list(raw_df['ticker'].unique())
if cfg.max_tickers > 0 and len(all_tickers) > cfg.max_tickers:
    rng = np.random.default_rng(cfg.random_state)
    all_tickers = rng.choice(all_tickers, cfg.max_tickers, replace=False).tolist()
    raw_df = raw_df[raw_df['ticker'].isin(all_tickers)].reset_index(drop=True)
    print(f'Subsampled to {cfg.max_tickers} tickers')
else:
    print(f'Using all {len(all_tickers)} tickers')

train_df, test_df = split_train_test_per_ticker(raw_df, cfg.test_size, cfg.target_horizon)
print(f'Train rows: {len(train_df):,} | Test rows: {len(test_df):,}')
folds = build_expanding_time_folds(train_df, cfg.n_splits, cfg.target_horizon)
print('CV folds:', len(folds))

cv_results = []
for fold_i, (fold_train, fold_val) in enumerate(folds, start=1):
    print(f'\n=== Fold {fold_i}/{len(folds)} ===')
    tasks = []
    for ticker in fold_val['ticker'].unique():
        tr_t = fold_train[fold_train['ticker'] == ticker].sort_values('date')
        va_t = fold_val[fold_val['ticker'] == ticker].sort_values('date')
        if len(tr_t) < 30 or len(va_t) < 2:
            continue
        tasks.append((tr_t, va_t))

    fold_outputs = Parallel(n_jobs=N_WORKERS, backend='loky')(
        delayed(run_prophet_for_ticker)(tr_t, va_t, cfg, False) for tr_t, va_t in tasks
    )
    fold_preds, fold_trues, fold_prevs = [], [], []
    for item in fold_outputs:
        if 'error' in item:
            continue
        fold_preds.extend(item['pred_close'].tolist())
        fold_trues.extend(item['actual_close'].tolist())
        fold_prevs.extend(item['prev_close'].tolist())

    if fold_trues:
        metrics = regression_metrics_with_direction(
            np.asarray(fold_trues), np.asarray(fold_preds), np.asarray(fold_prevs)
        )
        cv_results.append(metrics)
        print(metrics)
    else:
        print('No valid fold predictions')

cv_df = pd.DataFrame(cv_results)
if not cv_df.empty:
    print('\nCV mean metrics')
    print(cv_df.mean(numeric_only=True).to_string())
else:
    print('\nNo CV metrics were produced.')

print('\nTraining final per-ticker Prophet models...')
final_tasks = []
for ticker in test_df['ticker'].unique():
    tr_t = train_df[train_df['ticker'] == ticker].sort_values('date')
    te_t = test_df[test_df['ticker'] == ticker].sort_values('date')
    if len(tr_t) < 30 or len(te_t) < 2:
        continue
    final_tasks.append((tr_t, te_t))

final_outputs = Parallel(n_jobs=N_WORKERS, backend='loky')(
    delayed(run_prophet_for_ticker)(tr_t, te_t, cfg, True) for tr_t, te_t in final_tasks
)

all_preds, all_trues, all_prevs, all_meta = [], [], [], []
models_by_ticker = {}
for item in final_outputs:
    if 'error' in item:
        continue
    ticker = item['ticker']
    models_by_ticker[ticker] = item['model_json']
    test_ticker_df = test_df[test_df['ticker'] == ticker].sort_values('date')
    dates = test_ticker_df['date'].to_numpy()
    actual_close = item['actual_close']
    pred_close = item['pred_close']
    prev_close = item['prev_close']
    for idx in range(len(actual_close)):
        all_preds.append(float(pred_close[idx]))
        all_trues.append(float(actual_close[idx]))
        all_prevs.append(float(prev_close[idx]))
        all_meta.append((ticker, dates[idx], float(actual_close[idx])))

if not all_trues:
    raise RuntimeError('No Prophet models were trained successfully.')

y_true = np.asarray(all_trues, dtype=np.float64)
y_pred = np.asarray(all_preds, dtype=np.float64)
y_prev = np.asarray(all_prevs, dtype=np.float64)
test_metrics = regression_metrics_with_direction(y_true, y_pred, y_prev)
print('Held-out test metrics:', test_metrics)

payload = {
    'models_by_ticker': models_by_ticker,
    'regressor_cols': REGRESSOR_COLS,
    'config': asdict(cfg),
    'cv_results': cv_results,
    'test_metrics': test_metrics,
    'trained_tickers': sorted(models_by_ticker),
}
with open(cfg.model_output_path, 'wb') as f:
    pickle.dump(payload, f)
with open(cfg.config_output_path, 'w') as f:
    json.dump(asdict(cfg), f, indent=2)

pred_path = RESULTS_DIR / f'{RUN_TAG}-prophet_predictions.csv'
meta_df = pd.DataFrame(all_meta, columns=['ticker', 'date', 'close'])
meta_df['y_true'] = y_true
meta_df['y_pred'] = y_pred
meta_df['y_prev'] = y_prev
meta_df.to_csv(pred_path, index=False)

elapsed_min = (time.perf_counter() - start_time) / 60
print(f'\nSaved model      : {cfg.model_output_path}')
print(f'Saved config     : {cfg.config_output_path}')
print(f'Saved predictions: {pred_path}')
print(f'Trained tickers  : {len(models_by_ticker):,}')
print(f'Elapsed minutes  : {elapsed_min:.2f}')
